In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import time
import warnings
from collections import defaultdict

# 1. SILENCE WARNINGS & LOGS
warnings.filterwarnings("ignore", category=UserWarning)
lgb_params = {
    'n_estimators': 50,
    'min_child_samples': 1,
    'min_data_in_bin': 1,
    'verbosity': -1  # Silences the [LightGBM] Info/Warning logs
}

# 2. DATA & TRAINING
menu = [
    {"name": "Biryani", "category": "main", "price": 220},
    {"name": "Paneer Tikka", "category": "starter", "price": 180},
    {"name": "Butter Naan", "category": "bread", "price": 40},
    {"name": "Raita", "category": "side", "price": 50},
    {"name": "Gulab Jamun", "category": "dessert", "price": 90},
    {"name": "Cold Drink", "category": "drink", "price": 60},
    {"name": "Lassi", "category": "drink", "price": 70},
    {"name": "Chicken Kebab", "category": "starter", "price": 210}
]

orders = [
    {"items": ["Biryani", "Raita", "Cold Drink"]},
    {"items": ["Paneer Tikka", "Butter Naan", "Lassi"]},
    {"items": ["Biryani", "Gulab Jamun"]},
    {"items": ["Chicken Kebab", "Cold Drink"]},
    {"items": ["Biryani", "Raita"]},
    {"items": ["Paneer Tikka", "Lassi"]}
]

# Combo Mining
co_occurrence = defaultdict(lambda: defaultdict(int))
for order in orders:
    items = order['items']
    for i in items:
        for j in items:
            if i != j: co_occurrence[i][j] += 1

candidate_map = {item: sorted(neighbors, key=neighbors.get, reverse=True) 
                 for item, neighbors in co_occurrence.items()}

# Training Dataset
training_rows = []
all_names = [m['name'] for m in menu]
for order in orders:
    items = order['items']
    if len(items) < 2: continue
    target, cart = items[-1], items[:-1]
    
    for label, name in [(1, target), (0, np.random.choice([n for n in all_names if n not in items]))]:
        item_info = next(m for m in menu if m['name'] == name)
        training_rows.append({
            "cart_size": len(cart),
            "price": item_info['price'],
            "is_drink": 1 if item_info['category'] == 'drink' else 0,
            "label": label
        })

df_train = pd.DataFrame(training_rows)
model = lgb.LGBMClassifier(**lgb_params)
model.fit(df_train.drop('label', axis=1), df_train['label'])

# 3. PRODUCTION SERVICE CLASS
class RecommendationService:
    def __init__(self, lgb_model, candidates, menu_data):
        self.model = lgb_model
        self.candidates = candidates
        self.item_meta = {m['name']: m for m in menu_data}
        self.feature_names = ["cart_size", "price", "is_drink"]

    def get_recommendations(self, cart_items, top_n=3):
        last_item = cart_items[-1]
        if last_item not in self.candidates: return []
        
        cands = self.candidates[last_item]
        features = []
        for name in cands:
            item = self.item_meta[name]
            features.append({
                "cart_size": len(cart_items),
                "price": item['price'],
                "is_drink": 1 if item['category'] == 'drink' else 0
            })
        
        # FIX: Ensure feature names match the training set
        df_inf = pd.DataFrame(features)[self.feature_names]
        scores = self.model.predict_proba(df_inf)[:, 1]
        
        ranked = sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)
        return [r[0] for r in ranked if r[0] not in cart_items][:top_n]

# 4. FINAL STRESS TEST
def run_clean_stress_test(service, iterations=10000):
    all_items = list(service.item_meta.keys())
    latencies = []
    
    print(f"🚀 Running {iterations} tests without warnings...")
    start_suite = time.time()
    
    for _ in range(iterations):
        size = np.random.randint(1, 3)
        cart = np.random.choice(all_items, size=size, replace=False).tolist()
        
        t0 = time.perf_counter()
        service.get_recommendations(cart)
        latencies.append((time.perf_counter() - t0) * 1000)
    
    total_time = time.time() - start_suite
    print(f"\n--- 📊 Performance Report ---")
    print(f"🔹 Avg Latency: {np.mean(latencies):.2f} ms")
    print(f"🔹 P99 Latency:  {np.percentile(latencies, 99):.2f} ms")
    print(f"🔹 Total Time:   {total_time:.2f} seconds")
    print(f"🔹 Throughput:   {iterations/total_time:.0f} requests/sec")

recommender = RecommendationService(model, candidate_map, menu)
run_clean_stress_test(recommender)

🚀 Running 10000 tests without warnings...

--- 📊 Performance Report ---
🔹 Avg Latency: 2.40 ms
🔹 P99 Latency:  2.78 ms
🔹 Total Time:   25.00 seconds
🔹 Throughput:   400 requests/sec
